In [0]:
# ============================================================
# STEP 1 → INSTALL GOOGLE DRIVE LIBRARIES
# ============================================================

# Run once in Databricks

%pip install google-api-python-client google-auth google-auth-httplib2 google-auth-oauthlib


# ============================================================
# STEP 2 → IMPORT LIBRARIES
# ============================================================

import os
from pyspark.sql import SparkSession
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import io


# ============================================================
# STEP 3 → CREATE SPARK SESSION
# ============================================================

spark = SparkSession.builder.getOrCreate()


# ============================================================
# STEP 4 → DEFINE VARIABLES
# ============================================================

catalog_name = "dev"

# Unity Catalog Volume path
volume_path = f"/Volumes/{catalog_name}/staging/files/"

# Metadata table
metadata_table = f"{catalog_name}.staging.file_metadata"

# Google service account json
SERVICE_ACCOUNT_FILE = f"/Volumes/{catalog_name}/staging/metadata/service_account.json"

# Google Drive folder ID
folder_id = "YOUR_GOOGLE_DRIVE_FOLDER_ID"


# ============================================================
# STEP 5 → AUTHENTICATE GOOGLE DRIVE
# ============================================================

SCOPES = ['https://www.googleapis.com/auth/drive.readonly']

credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=SCOPES
)

service = build('drive', 'v3', credentials=credentials)


# ============================================================
# STEP 6 → CREATE METADATA TABLE
# ============================================================

spark.sql(f"""

CREATE TABLE IF NOT EXISTS {metadata_table} (

    file_id STRING,
    file_name STRING,
    drive_modified_time TIMESTAMP,
    downloaded_time TIMESTAMP

)
USING DELTA

""")


# ============================================================
# STEP 7 → CREATE INCREMENTAL DOWNLOAD FUNCTION
# ============================================================

def incremental_download(file_id,
                         file_name,
                         drive_modified_time):

    # --------------------------------------------------------
    # STEP 7.1 → CHECK IF FILE ALREADY EXISTS
    # --------------------------------------------------------

    existing = spark.sql(f"""
        SELECT drive_modified_time
        FROM {metadata_table}
        WHERE file_id = '{file_id}'
    """).collect()

    should_download = False


    # --------------------------------------------------------
    # STEP 7.2 → NEW FILE
    # --------------------------------------------------------

    if len(existing) == 0:

        print(f"🆕 New file → {file_name}")

        should_download = True


    # --------------------------------------------------------
    # STEP 7.3 → MODIFIED FILE
    # --------------------------------------------------------

    else:

        old_modified_time = existing[0]["drive_modified_time"]

        if str(drive_modified_time) > str(old_modified_time):

            print(f"♻️ Modified file → {file_name}")

            should_download = True

        else:

            print(f"⏭️ Skipping unchanged file → {file_name}")


    # --------------------------------------------------------
    # STEP 7.4 → DOWNLOAD FILE
    # --------------------------------------------------------

    if should_download:

        request = service.files().get_media(fileId=file_id)

        file_stream = io.BytesIO()

        downloader = MediaIoBaseDownload(file_stream, request)

        done = False

        while done is False:

            status, done = downloader.next_chunk()

        # ----------------------------------------------------
        # STEP 7.5 → SAVE FILE INTO VOLUME
        # ----------------------------------------------------

        file_path = os.path.join(volume_path, file_name)

        with open(file_path, "wb") as f:

            f.write(file_stream.getvalue())

        print(f"✅ Downloaded → {file_name}")


        # ----------------------------------------------------
        # STEP 7.6 → UPDATE METADATA TABLE
        # ----------------------------------------------------

        spark.sql(f"""
            DELETE FROM {metadata_table}
            WHERE file_id = '{file_id}'
        """)

        spark.sql(f"""

            INSERT INTO {metadata_table}
            VALUES (

                '{file_id}',
                '{file_name}',
                TIMESTAMP('{drive_modified_time}'),
                current_timestamp()

            )

        """)

        print(f"✅ Metadata updated → {file_name}")


# ============================================================
# STEP 8 → READ FILES FROM GOOGLE DRIVE
# ============================================================

results = service.files().list(

    q=f"'{folder_id}' in parents and trashed = false",

    fields="files(id, name, modifiedTime)"

).execute()

files = results.get('files', [])


# ============================================================
# STEP 9 → PROCESS EACH FILE
# ============================================================

for file in files:

    file_id = file['id']

    file_name = file['name']

    modified_time = file['modifiedTime']

    print(f"\nProcessing → {file_name}")

    incremental_download(
        file_id,
        file_name,
        modified_time
    )


# ============================================================
# STEP 10 → COMPLETED
# ============================================================

print("\n✅ Incremental ingestion completed")